# SNP genotyping from FLASH-merged sequencing reads (from bulk gDNA)

This notebook performs genotyping analysis for a single sample used in Fig. S4A from a FLASH-merged FASTQ file (`*.extendedFrags.fastq.gz`).

## Overview

The workflow:
1. Reads a merged FASTQ file
2. Extracts sequences between predefined upstream and downstream regions flanking the SNP
3. Filters reads containing both flanking sequences
4. Quantifies sequence frequencies
5. Identifies the most abundant sequences

## Input data

Input file is a FLASH-merged FASTQ file: `<sample_name>.extendedFrags.fastq.gz`

Example:
- `T2K_gDNA.extendedFrags.fastq.gz`

The file should be located in the specified working directory.

## Sequence extraction

Upstream and downstream sequences (12nt each) define the region surrounding the SNP (±20 bp).

Only reads containing both flanking sequences are retained.  
The sequence between these regions is extracted for downstream analysis.

## Summary metrics

The following metrics are calculated:
- Total number of reads
- Percentage of reads containing both flanking sequences
- Sequence frequency table
- Relative abundance (%) of each sequence

## Output

The notebook generates:
- A frequency table of extracted sequences ranked by abundance

Results are saved as:
`T80K_gDNA_20bp.csv`

## Notes

- Paired-end reads were merged prior to analysis using FLASH.


In [1]:
suppressPackageStartupMessages({
    library("Biostrings")
    library("stringr")
    library("dplyr")
    })

In [4]:
#input data
WD <- "path/to/data/"
sample_name <- "T2K_gDNA"
input <- paste(WD, sample_name, ".extendedFrags.fastq.gz", sep = "")
df <- data.frame(readDNAStringSet(input, format="fastq", use.names=FALSE))
colnames(df) <- c('sequence')
head(df)

,sequence
,<chr>
1,CGACGGAAAGAGTATGAGCTGGAAAAACAGAAAAAACTTGAAAAGGAAAGACAAGAACAACTCCAAAAGGAGCAAGAGAAAGCCTTTTTCGCTCAGTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTCAGCCAGCCCAGCACATCCAGTCAGAAACCAGTGGATCTGCCAACTACTCCCAGGTACAGAGT
2,CGACGGAAGAGTATGAGCTGGAAAAACAGAAAAAACTTGAAAAGGAAAGACAAGAACAACTCCAAAAGGAGCAAGAGAAAGCCTTTTTCGCTCAGTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTCAGCCAGCCCAGCACATCCAGTCAGAAACCAGTGGATCTGCCAACTACTCCCAGGTACAGAGT
3,CGACGGAAAGAGTATGAGCTGGAAAAACAGAAAAAACTTGAAAAGGAAAGACAAGAACAACTCCAAAAGGAGCAAGAGAAAGCCTTTTTCGCTCAGTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTCAGCCAGCCCAGCACATCCAGTCAGAAACCAGTGGATCTGCCAACTACTCCCAGGTACAGAGT
4,CGACGGAAAGAGTATGAGCTGGAAAAACAGAAAAAACTTGAAAAGGAAAGACAAGAACAACTCCAAAAGGAGCAAGAGAAAGCCTTTTTCGCTCAGTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTCAGCCAGCCCAGCACATCCAGTCAGAAACCAGTGGATCTGCCAACTACTCCCAGGTACAGAGT
5,CGACGGAAAGAGTATGAGCTGGAAAAACAGAAAAAACTTGAAAAGGAAAGACAAGAACAACTCCAAAAGGAGCAAGAGAAAGCCTTTTTCGCTCAGTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTCAGCCAGCCCAGCACATCCAGTCAGAAACCAGTGGATCTGCCAACTACTCCCAGGTACAGAGT
6,CGACGGAAAGAGTATGAGCTGGAAAAACAGAAAAAACTTGAAAAGGAAAGACAAGAACAACTCCAAAAGGAGCAAGAGAAAGCCTTTTTCGCTCAGTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTCAGCCAGCCCAGCACATCCAGTCAGAAACCAGTGGATCTGCCAACTACTCCCAGGTACAGAGT


In [5]:
#upstream or downstream sequences (12nt each) of the editing window (±20 from the SNP position)
upstream_seq <- "CTTTTTCGCTCA"

downstream_seq <- "AGCCAGCCCAGC"

In [6]:
#extract sequences between upstream_seq and downstream_seq
up <- data.frame(str_split_fixed(df$sequence, upstream_seq, 2))
colnames(up) <- c("split_1", "split_2")
up[up == ""] <- NA #convert blank into NA
up <- na.omit(up) #if the upstream_seq was not detected, remove

down <- data.frame(str_split_fixed(up$split_2, downstream_seq, 2))
colnames(down) <- c("split_1", "split_2")
down[down == ""] <- NA #convert blank into NA
down <- na.omit(down) #if the downstream_seq was not detected, remove

df2 <- data.frame("sequence" = down$split_1)
head(df2)
dim(df2)

,sequence
,<chr>
1,GTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTC
2,GTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTC
3,GTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTC
4,GTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTC
5,GTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTC
6,GTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTC


[1] 10913331        1

In [7]:
# % of reads that contain up and downstream sequences.
nrow(df2)/nrow(df)*100

[1] 98.34412

In [9]:
#summarise results
result <- data.frame(table(unlist(df2$sequence)))
colnames(result) <- c("sequence", "frequency")
result <- result[order(result$frequency, decreasing=TRUE),]
result$ratio <- round(result$frequency/nrow(df2)*100, 2)
head(result, 30)

,sequence,frequency,ratio
,<fct>,<int>,<dbl>
332,GTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATTC,10053711,92.12
185,GTTACAACTAGATGAAGAGAAAGGTGAATTTCTCCCAATTC,731082,6.70
326,GTTACAACTAGATGAAGAGACAGGTGAATTTCTCCCAATGC,32262,0.30
388,GTTACAACTAGATGAAGAGACAGGTGAATTTTTCCCAATTC,11538,0.11
370,GTTACAACTAGATGAAGAGACAGGTGAATTTCTTCCAATTC,11379,0.10
451,GTTACAACTAGATGAAGAGACGGGTGAATTTCTCCCAATTC,5923,0.05
372,GTTACAACTAGATGAAGAGACAGGTGAATTTCTTCCCAATTC,4500,0.04
121,GTTACAACTAGATGAAGACAGGTGAATTTCTCCCAATTC,3078,0.03
381,GTTACAACTAGATGAAGAGACAGGTGAATTTTCTCCCAATTC,3042,0.03


In [13]:
write.csv(result, file = "T80K_gDNA_20bp.csv", row.names = FALSE)

In [14]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /software/cellgen/team205/si9/envs/Seurat/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
[1] dplyr_1.1.4         stringr_1.5.2       Biostrings_2.70.3  
[4] GenomeInfoDb_1.38.8 XVector_0.42.0      IRanges_2.36.0     
[7] S4Vectors_0.40.2    BiocGenerics_0.48.1

loaded via a namespace (and not attached):
 [1] crayon_1.5.3            vctrs_0.6.5             cli_3.6.5              
 [4] 